In [1]:
import os, getpass
from dotenv import load_dotenv
load_dotenv()

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("AZURE_OPENAI_API_KEY")
_set_env("AZURE_OPENAI_ENDPOINT")

from autogen_ext.models.openai import AzureOpenAIChatCompletionClient
# Define a model client. You can use other model client that implements
# the `ChatCompletionClient` interface.
model_client = AzureOpenAIChatCompletionClient(
    model="gpt-4o",        
    api_version="2024-10-21",
    temperature=0
)

In [2]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console



In [3]:


planner_agent = AssistantAgent(
    "planner_agent",
    model_client=model_client,
    description="A helpful assistant that can plan trips.",
    system_message="You are a helpful assistant that can suggest a travel plan for a user based on their request.",
)

local_agent = AssistantAgent(
    "local_agent",
    model_client=model_client,
    description="A local assistant that can suggest local activities or places to visit.",
    system_message="You are a helpful assistant that can suggest authentic and interesting local activities or places to visit for a user and can utilize any context information provided.",
)

language_agent = AssistantAgent(
    "language_agent",
    model_client=model_client,
    description="A helpful assistant that can provide language tips for a given destination.",
    system_message="You are a helpful assistant that can review travel plans, providing feedback on important/critical tips about how best to address language or communication challenges for the given destination. If the plan already includes language tips, you can mention that the plan is satisfactory, with rationale.",
)

travel_summary_agent = AssistantAgent(
    "travel_summary_agent",
    model_client=model_client,
    description="A helpful assistant that can summarize the travel plan.",
    system_message="You are a helpful assistant that can take in all of the suggestions and advice from the other agents and provide a detailed final travel plan. You must ensure that the final plan is integrated and complete. YOUR FINAL RESPONSE MUST BE THE COMPLETE PLAN. When the plan is complete and all perspectives are integrated, you can respond with TERMINATE.",
)


In [4]:
termination = TextMentionTermination("TERMINATE")
group_chat = RoundRobinGroupChat(
    [planner_agent, local_agent, language_agent, travel_summary_agent], termination_condition=termination
)
await Console(group_chat.run_stream(task="Plan a 3 day trip to Nepal."))

await model_client.close()


---------- TextMessage (user) ----------
Plan a 3 day trip to Nepal.
---------- TextMessage (planner_agent) ----------
Nepal is a beautiful country with stunning landscapes, rich culture, and warm hospitality. A 3-day trip will give you a glimpse of its charm, focusing on Kathmandu Valley and nearby attractions. Here's a suggested itinerary:

---

### **Day 1: Arrival in Kathmandu & Exploring the City**
- **Morning: Arrival in Kathmandu**
  - Land at Tribhuvan International Airport.
  - Check into your hotel in Thamel, the bustling tourist hub of Kathmandu.
  - Freshen up and enjoy a traditional Nepali welcome drink (masala tea or lassi).

- **Afternoon: Explore UNESCO World Heritage Sites**
  - **Swayambhunath Stupa (Monkey Temple):** Visit this iconic Buddhist stupa perched on a hill, offering panoramic views of Kathmandu Valley.
  - **Kathmandu Durbar Square:** Explore the ancient royal palace, temples, and courtyards. Don’t miss the Kumari Ghar, home to the living goddess.

- **Eve

c:\Users\zacharyhou\miniconda3\envs\agent-ready\Lib\site-packages\autogen_agentchat\agents\_assistant_agent.py:955: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


---------- TextMessage (local_agent) ----------
That sounds like a fantastic plan! If you'd like to add a bit of adventure or relaxation to your trip, here are a few optional activities you can consider:

- **Adventure Option:** On Day 3, instead of Nagarkot, you could opt for an early morning mountain flight to see Mount Everest up close. These flights typically last an hour and offer stunning aerial views of the Himalayas.

- **Relaxation Option:** If you prefer a slower pace, you could spend your last afternoon at a spa in Kathmandu, enjoying a traditional Ayurvedic massage or a Himalayan salt therapy session.

Let me know if you'd like to tweak the itinerary further or need recommendations for accommodations or restaurants!
---------- TextMessage (language_agent) ----------
This is a well-thought-out itinerary for a 3-day trip to Nepal, covering cultural landmarks, scenic views, and local experiences. However, I noticed that language and communication tips are not explicitly addres